In [1]:
# ============================================================
# VEHICLE CLASS-WISE SPEED DATA CLEANING
# OUTLIER REMOVAL + NEW V85 CALCULATION
# INPUT: NN initial data
# OUTPUT: NN cleaned data
# ============================================================

import pandas as pd
import numpy as np
import os
from pathlib import Path


# ============================================================
# STEP 1: Define input and output folders
# ============================================================

input_folder = r"D:\MSC THESIS AAVAS\Vehicle class wise\initial data\NNM\NN"

output_folder = r"D:\MSC THESIS AAVAS\Vehicle class wise\Cleaned data\NNM\NN"

# Create output folder if it does not exist
os.makedirs(output_folder, exist_ok=True)

print("Input folder:", input_folder)
print("Output folder:", output_folder)


# ============================================================
# STEP 2: Function to clean each sheet
# ============================================================

def clean_speed_data(df):
    """
    For each sheet:
    1. Removes extra spaces from column names
    2. Removes outliers using IQR method
    3. Keeps only accepted speed values
    4. Calculates new 85th percentile speed from accepted values only
    5. Returns only:
       Chainage | Speed | Unit | New_85th_Percentile
    """

    # Remove extra spaces from column names
    df.columns = df.columns.astype(str).str.strip()

    # Check required columns
    if "Chainage" not in df.columns:
        raise ValueError("Chainage column not found")

    if "Speed" not in df.columns:
        raise ValueError("Speed column not found")

    # Convert speed to numeric
    df["Speed"] = pd.to_numeric(df["Speed"], errors="coerce")

    # Remove blank or non-numeric speed values
    df = df.dropna(subset=["Speed"]).copy()

    # Calculate Q1, Q3 and IQR
    Q1 = df["Speed"].quantile(0.25)
    Q3 = df["Speed"].quantile(0.75)
    IQR = Q3 - Q1

    # Outlier limits
    lower_limit = Q1 - 1.5 * IQR
    upper_limit = Q3 + 1.5 * IQR

    # Keep only accepted speeds
    cleaned_df = df[
        (df["Speed"] >= lower_limit) &
        (df["Speed"] <= upper_limit)
    ].copy()

    # Calculate new V85 from accepted speed values only
    if len(cleaned_df) > 0:
        new_v85 = np.percentile(cleaned_df["Speed"], 85)
    else:
        new_v85 = np.nan

    # Add Unit column
    cleaned_df["Unit"] = "km/hr"

    # Add new V85 column
    cleaned_df["New_85th_Percentile"] = round(new_v85, 2)

    # Keep only required columns
    cleaned_df = cleaned_df[
        ["Chainage", "Speed", "Unit", "New_85th_Percentile"]
    ]

    # Summary information
    summary = {
        "Original_Count": len(df),
        "Cleaned_Count": len(cleaned_df),
        "Outliers_Removed": len(df) - len(cleaned_df),
        "Q1": Q1,
        "Q3": Q3,
        "IQR": IQR,
        "Lower_Limit": lower_limit,
        "Upper_Limit": upper_limit,
        "New_85th_Percentile": new_v85
    }

    return cleaned_df, summary


# ============================================================
# STEP 3: Process all Excel files and all sheets
# ============================================================

summary_list = []

# Read all Excel files from input folder
excel_files = list(Path(input_folder).glob("*.xlsx"))

# Avoid temporary Excel files
excel_files = [
    file for file in excel_files
    if not file.name.startswith("~$")
]

print(f"Total Excel files found: {len(excel_files)}")

for file_path in excel_files:

    file_name = file_path.name
    print(f"\nProcessing file: {file_name}")

    # Read all sheets from current Excel file
    all_sheets = pd.read_excel(file_path, sheet_name=None)

    cleaned_sheets = {}

    for sheet_name, df in all_sheets.items():

        print(f"  Processing sheet: {sheet_name}")

        try:
            cleaned_df, summary = clean_speed_data(df)

            cleaned_sheets[sheet_name] = cleaned_df

            summary_list.append({
                "File_Name": file_name,
                "Sheet_Name": sheet_name,
                "Original_Count": summary["Original_Count"],
                "Cleaned_Count": summary["Cleaned_Count"],
                "Outliers_Removed": summary["Outliers_Removed"],
                "Q1": round(summary["Q1"], 2),
                "Q3": round(summary["Q3"], 2),
                "IQR": round(summary["IQR"], 2),
                "Lower_Limit": round(summary["Lower_Limit"], 2),
                "Upper_Limit": round(summary["Upper_Limit"], 2),
                "New_85th_Percentile": round(summary["New_85th_Percentile"], 2)
            })

        except Exception as e:
            print(f"    Skipped sheet '{sheet_name}' because: {e}")

    # Save cleaned workbook in output folder
    cleaned_output_path = os.path.join(
        output_folder,
        "Cleaned_" + file_name
    )

    if cleaned_sheets:
        with pd.ExcelWriter(cleaned_output_path, engine="openpyxl") as writer:
            for sheet_name, data in cleaned_sheets.items():
                data.to_excel(writer, sheet_name=sheet_name[:31], index=False)

        print(f"  Saved: {cleaned_output_path}")

print("\nAll Excel files cleaned successfully.")


# ============================================================
# STEP 4: Save final V85 summary table
# ============================================================

summary_df = pd.DataFrame(summary_list)

summary_output_path = os.path.join(
    output_folder,
    "Final_V85_Summary_After_Outlier_Removal.xlsx"
)

summary_df.to_excel(summary_output_path, index=False)

print("\nFinal summary saved at:")
print(summary_output_path)

summary_df

Input folder: D:\MSC THESIS AAVAS\Vehicle class wise\initial data\NNM\NN
Output folder: D:\MSC THESIS AAVAS\Vehicle class wise\Cleaned data\NNM\NN
Total Excel files found: 3

Processing file: Heavy Vehicle.xlsx
  Processing sheet: Sheet1
  Processing sheet: Sheet2
  Processing sheet: Sheet3
  Processing sheet: Sheet4
  Processing sheet: Sheet5
  Processing sheet: Sheet6
  Processing sheet: Sheet7
  Processing sheet: Sheet8
  Processing sheet: Sheet9
  Processing sheet: Sheet10
  Processing sheet: Sheet11
  Processing sheet: Sheet12
  Processing sheet: Sheet13
  Processing sheet: Sheet14
  Processing sheet: Sheet15
  Processing sheet: Sheet16
  Processing sheet: Sheet18
  Processing sheet: Sheet19
  Processing sheet: Sheet20
  Processing sheet: Sheet21
  Processing sheet: Sheet22
  Processing sheet: Sheet23
  Processing sheet: Sheet24
  Processing sheet: Sheet25
  Processing sheet: Sheet26
  Processing sheet: Sheet27
  Processing sheet: Sheet28
  Processing sheet: Sheet29
  Processing s

,File_Name,Sheet_Name,Original_Count,Cleaned_Count,Outliers_Removed,Q1,Q3,IQR,Lower_Limit,Upper_Limit,New_85th_Percentile
0,Heavy Vehicle.xlsx,Sheet1,34,34,0,30.25,36.0,5.75,21.62,44.62,42.00
1,Heavy Vehicle.xlsx,Sheet2,33,33,0,29.00,35.0,6.00,20.00,44.00,37.20
2,Heavy Vehicle.xlsx,Sheet3,20,20,0,31.00,34.0,3.00,26.50,38.50,34.15
3,Heavy Vehicle.xlsx,Sheet4,26,24,2,28.25,32.0,3.75,22.62,37.62,32.00
4,Heavy Vehicle.xlsx,Sheet5,35,35,0,30.50,39.5,9.00,17.00,53.00,40.90
...,...,...,...,...,...,...,...,...,...,...,...
100,Two Wheelers.xlsx,Sheet32,21,21,0,33.00,40.0,7.00,22.50,50.50,45.00
101,Two Wheelers.xlsx,Sheet33,20,20,0,34.00,38.0,4.00,28.00,44.00,39.00
102,Two Wheelers.xlsx,Sheet34,24,24,0,29.00,37.0,8.00,17.00,49.00,39.10
103,Two Wheelers.xlsx,Sheet35,27,24,3,28.00,30.5,2.50,24.25,34.25,31.55
